# Name That Model - Training Pipeline

This notebook trains a model to identify the authorship of AI-generated text passages among 18 candidate frontier language models.

The task: Given a text passage, predict which of the 18 models generated it.

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, balanced_accuracy_score
import re
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded')

Libraries loaded


## 1. Load Training Data

In [2]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print(f'Training samples: {len(train)}')
print(f'Test samples: {len(test)}')
print('\nModel distribution:')
print(train['model'].value_counts())

Training samples: 2699
Test samples: 3

Model distribution:
model
qwen3.8-max               150
claude-sonnet-5           150
gemini-3.6-flash          150
gpt-5.6-terra             150
glm-5.3-flash             150
gemini-3.1-pro-preview    150
gemini-3.8-flash          150
claude-opus-4-6           150
gpt-5.6-luna              150
claude-opus-4-8           150
gemini-3.7-flash          150
claude-opus-5             150
kimi-k3                   150
gpt-5.5                   150
gpt-6-astra               150
glm-5.3                   150
gpt-5.6-sol               150
claude-fable-5            149
Name: count, dtype: int64


## 2. Text Preprocessing

In [3]:
def clean_text(text):
    """Basic text cleaning: lowercase, remove punctuation, collapse whitespace."""
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train['clean_text'] = train['text'].astype(str).apply(clean_text)
test['clean_text'] = test['text'].astype(str).apply(clean_text)

print(f"Cleaned training samples: {len(train)}")
print(f"Cleaned test samples: {len(test)}")

Cleaned training samples: 2699
Cleaned test samples: 3


## 3. Train Classifier

In [4]:
# Encode labels
label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(train['model'])

print(f'Number of classes: {len(set(train["model"]))}')
print(f'Class distribution:\n{train["model"].value_counts()}')

# Build pipeline with TF-IDF + Logistic Regression
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=5000,
        ngram_range=(1, 2),
        stop_words='english'
    )),
    ('clf', LogisticRegression(
        max_iter=1000,
        solver='lbfgs'
    ))
])

# Train
print('Training classifier...')
pipeline.fit(train['clean_text'], encoded_labels)
print('Training complete')

Number of classes: 18
Class distribution:
model
qwen3.8-max               150
claude-sonnet-5           150
gemini-3.6-flash          150
gpt-5.6-terra             150
glm-5.3-flash             150
gemini-3.1-pro-preview    150
gemini-3.8-flash          150
claude-opus-4-6           150
gpt-5.6-luna              150
claude-opus-4-8           150
gemini-3.7-flash          150
claude-opus-5             150
kimi-k3                   150
gpt-5.5                   150
gpt-6-astra               150
glm-5.3                   150
gpt-5.6-sol               150
claude-fable-5            149
Name: count, dtype: int64
Training classifier...


Training complete


## 4. Evaluate on Training Set

In [5]:
# Predict on training set
train_preds = pipeline.predict(train['clean_text'])
train_acc = accuracy_score(train_preds, encoded_labels)
print(f'Training accuracy: {train_acc:.4f}')

# Balanced accuracy
bal_acc = balanced_accuracy_score(encoded_labels, train_preds)
print(f'Balanced accuracy: {bal_acc:.4f}')

results = classification_report(encoded_labels, train_preds, target_names=label_encoder.classes_)
print(results)

Training accuracy: 0.9856
Balanced accuracy: 0.9855
                        precision    recall  f1-score   support

        claude-fable-5       0.99      0.98      0.98       149
       claude-opus-4-6       0.99      0.99      0.99       150
       claude-opus-4-8       0.99      0.98      0.99       150
         claude-opus-5       0.97      1.00      0.98       150
       claude-sonnet-5       1.00      0.97      0.99       150
gemini-3.1-pro-preview       0.99      1.00      0.99       150
      gemini-3.6-flash       0.97      0.99      0.98       150
      gemini-3.7-flash       0.99      0.99      0.99       150
      gemini-3.8-flash       0.99      0.98      0.98       150
               glm-5.3       0.99      0.99      0.99       150
         glm-5.3-flash       0.99      0.99      0.99       150
               gpt-5.5       0.99      0.99      0.99       150
          gpt-5.6-luna       0.99      1.00      1.00       150
           gpt-5.6-sol       0.99      0.97      0.

## 5. Predict on Test Set & Create Submission

In [6]:
# Predict on test data
test_preds = pipeline.predict(test['clean_text'])
test_labels = label_encoder.inverse_transform(test_preds)

# Create submission dataframe
submission = pd.DataFrame({
    'id': test['id'].astype(str),
    'model': test_labels
})

# Ensure model names are valid
valid_models = [
    'gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra',
    'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash',
    'claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5',
    'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3'
]
submission['model'] = submission['model'].apply(lambda x: x if x in valid_models else 'gpt-5.5')

# Ensure no null values and no duplicates in id
submission = submission.drop_duplicates(subset=['id'], keep='first')
submission = submission.dropna(subset=['id', 'model'])

# Save submission as CSV with header id,model
submission.to_csv('submission.csv', index=False)

print('Submission created: submission.csv')
print(submission.head(10))

Submission created: submission.csv
             id             model
0  00353fb1c541     claude-opus-5
1  005f5a9a6139   claude-sonnet-5
2  0062a53d5487  gemini-3.7-flash
